# ContractRisk — candidato OCR 19 (CPU)

La prueba visual 18 falló en utilidad (0/5 legibles). Este candidato **nuevo y separado** usa Tesseract 5, idiomas spa+eng y dos segmentaciones fijas (3 y 6). Se conservan los resultados Qwen: no se cambia su prompt ni se reemplazan cifras anteriores. La concordancia entre dos segmentaciones del mismo motor no es verificación independiente y su confianza no es probabilidad calibrada.

**No necesita GPU: seleccionar hardware Ninguno.** No necesita claves, Hugging Face ni API pagada. Una pasada sobre doce imágenes ya congeladas: ocho previas y cuatro sintéticas nuevas. La prueba evalúa lectura de píxeles, no respuestas QA, RAG, fuente contractual o calidad del despliegue. No cambia main ni el Space.

Ejecutar en orden; si falla, descargar evidencia desde la última celda. No repetir para buscar un acierto.

In [ ]:
import os,sys,subprocess,tempfile,json
from pathlib import Path
SETUP_OK=False
ROOT=Path(tempfile.mkdtemp(prefix='contractrisk_ocr19_',dir='/content'))/'repo'
subprocess.run(['git','clone','--single-branch','--branch','Entrega/contractrisk-final','https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git',str(ROOT)],check=True)
os.chdir(ROOT)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
print('Código:',COMMIT)
def run_visible(arguments,logfile):
    with open(logfile,'w',encoding='utf-8') as log:
        process=subprocess.Popen(arguments,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        status=process.wait()
    if status:raise RuntimeError('Proceso detenido; descargar diagnóstico con la última celda: '+logfile)


In [ ]:
SETUP_OK=False
run_visible([sys.executable,'-m','pip','install','pillow==11.3.0'],'ocr19_install.log')
run_visible(['apt-get','update'],'ocr19_apt.log')
run_visible(['apt-get','install','-y','tesseract-ocr','tesseract-ocr-spa','tesseract-ocr-eng'],'ocr19_engine_install.log')
run_visible(['tesseract','--version'],'ocr19_engine_version.log')
run_visible(['tesseract','--list-langs'],'ocr19_languages.log')
run_visible([sys.executable,'-m','unittest','discover','-s','final/tests','-p','test_ocr19.py','-v'],'ocr19_tests.log')
from final.ocr_probe19 import load_protocol
print('Doce entradas congeladas:',[c['id'] for c in load_protocol()['cases']])
SETUP_OK=True


In [ ]:
if not SETUP_OK:raise RuntimeError('Preparación incompleta. Descargar diagnóstico sin ejecutar OCR.')
RESULTS=ROOT/'ocr19_results'
run_visible([sys.executable,'-u','final/ocr_probe19.py','--output',str(RESULTS)],'ocr19_readings.log')


## Interpretación y evidencia

Una lectura exacta no confirma autenticidad ni correspondencia con el corpus. En el caso de valor alterado, leer la cifra alterada es correcto como OCR pero NO como respuesta sobre el snapshot. `needs_source_and_answer_validation` significa que esas validaciones siguen pendientes. Un resultado con campo ausente debe bloquear preguntas sobre ese campo.

`completed_requires_review` no aprueba la integración. Guardamos las dos lecturas TSV, confianza, campos y rechazos. No hay búsqueda de parámetros ni reparación de dígitos a partir de referencias. Enviar ZIP y este notebook con salidas para revisión antes de integrar o desplegar.

In [ ]:
import zipfile
from google.colab import files
metadata=ROOT/'ocr19_results/metadata.json'
print(metadata.read_text(encoding='utf-8') if metadata.exists() else 'No comenzó OCR o falta metadata. Revisar logs conservados.')
archive=ROOT.parent/'contractrisk_final_ocr19_resultados.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for folder in ['ocr19_results','final/evidence/ocr19_protocol','final/evidence/visual18_protocol']:
        base=ROOT/folder
        if base.exists():
            for item in sorted(base.rglob('*')):
                if item.is_file():z.write(item,item.relative_to(ROOT))
    for item in sorted(ROOT.glob('ocr19_*.log')):z.write(item,item.name)
    z.writestr('checkout_commit.txt',COMMIT+'\n')
    z.writestr('setup_state.json',json.dumps({'setup_ok':SETUP_OK}))
files.download(str(archive))
